### Load the dataset

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/tharunnarra/house-price-prediction-dataset/HousePricePrediction - Sheet1.csv')

### Observe the dataset

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

### Drop duplicate rows if present

In [ ]:
df.drop_duplicates()
df.shape

### Droping Id column

In [ ]:
df.drop(columns=['Id'],inplace=True)
df.head()

### Checking for missing values

In [ ]:
df.isna().sum()

### Replacing the missing values in the saleprice column with the mean.

In [ ]:
df['SalePrice'].mean()

In [ ]:
df['SalePrice'] = df['SalePrice'].fillna(df['SalePrice'].mean())

In [ ]:
df.isna().sum()

### Replacing remaining missing values with 0.

In [ ]:
df['MSZoning'] = df['MSZoning'].fillna("Missing")
df['Exterior1st'] = df['Exterior1st'].fillna("Missing")
df.fillna(0,inplace=True)
df.isna().sum()

### Handling outliers using IQR.

In [ ]:
df.describe()

In [ ]:
import numpy as np
Q1 = np.percentile(df['LotArea'],25,interpolation = 'midpoint')
Q3 = np.percentile(df['LotArea'],75,interpolation = 'midpoint')
print(f"Q1 = {Q1}")
print(f"Q3 = {Q3}")

In [ ]:
IQR = Q3 - Q1
lowerbound = Q1 - 1.5*IQR
upperbound = Q3 + 1.5*IQR
print(f"lower bound for lot-area = {lowerbound}")
print(f"upper bound for lot-area = {upperbound}")

In [ ]:
df = df[(df['LotArea'] < upperbound) & (df['LotArea'] > lowerbound)]

In [ ]:
df.shape

### Handling categorical model

In [ ]:
df.info()

In [ ]:
categorical_cols = df.select_dtypes('object').columns.tolist()
categorical_cols

In [ ]:
print(df['MSZoning'].unique())
print(df['LotConfig'].unique())
print(df['BldgType'].unique())
print(df['Exterior1st'].unique())

In [ ]:
df = pd.get_dummies(df,columns=categorical_cols,dtype=int,drop_first=True)

In [ ]:
df.head()

In [ ]:
df.columns

### Splitting the data

In [ ]:
df = df.sample(frac=1,random_state=42).reset_index(drop=True)
split_index = int(0.8*len(df))

train = df[:split_index]
test = df[split_index:]

In [ ]:
print(train.shape)
print(test.shape)

In [ ]:
x_train = train.drop(columns='SalePrice')
y_train = train['SalePrice']
x_test = test.drop(columns = 'SalePrice')
y_test = test['SalePrice']

### Scaling the data

In [ ]:
mean = x_train.mean()
std = x_train.std()


In [ ]:
x_train = (x_train - mean)/std
x_test = (x_test - mean)/std

### My Linear Regression Model

In [ ]:
class Multiple_LR_Model:
    def __init__(self, learning_rate=0.01, iterations=1000):
        """
        Initialize the Linear Regression model.

        Args:
            learning_rate : Learning rate.Defaults to 0.01.
            iterations : Number of iteratoins for training.Defaults to 1000.
        """
        self.learning_rate = learning_rate
        self.interations = iterations
        self.w = None
        self.b = None

    def cost_function(self, x, y, w, b):
        """
        Compute the cost(squared error).

        Args:
            x (ndarray): Data, m examples with n features. Shape(m,n).
            y (ndarray): Target values. Shape(m,).
            w (ndarray): Model weights. Shape(n,).
            b (float): Model bias.

        Returns:
            cost (float): cost(sqaured error)
        """
        m = x.shape[0]
        cost = 0
        for i in range(m):
            fw_b = np.dot(x[i], w) + b
            cost += (fw_b - y[i]) ** 2
        total = cost / (2 * m)
        return total

    def compute_gradient(self, x, y, w, b):
        """
        Computes the gradient for linear regression.

        Args:
            x (ndarray): Data, m examples with n features. Shape(m,n).
            y (ndarray): Target values. Shape(m,).
            w (ndarray): Model weights. Shape(n,).
            b (float): Model bias.

        Returns:
            dj_dw : The gradient of the cost w.r.t. the weights. Shape(n,).
            dj_db : The gradient of the cost w.r.t. the bias.
        """
        m, n = x.shape
        dj_db = 0
        dj_dw = np.zeros((n,))
        for i in range(m):
            fw_b = np.dot(x[i], w) + b
            for j in range(n):
                dj_dw[j] += (fw_b - y[i]) * x[i, j]
            dj_db += fw_b - y[i]
        dj_dw = dj_dw / m
        dj_db = dj_db / m
        return dj_dw, dj_db

    def train_model(self, x, y):
        """
        Trains the model to the training data using gradient descent.

        Args:
            x (ndarray): Training data. Shape (m,n).
            y (ndarray): Traget values. Shape (m,).

        Returns:
            w (ndarray): Final wegihts calculated by gradient descent.
            b (float): Final bias calculated by gradient descent.
        """
        _, n = x.shape
        self.w = np.zeros(n)
        self.b = 0
        print(f"Initial cost :{self.cost_function(x, y, self.w, self.b):.4f}")
        for i in range(self.interations):
            dj_dw, dj_db = self.compute_gradient(x, y, self.w, self.b)
            self.w = self.w - self.learning_rate * dj_dw
            self.b = self.b - self.learning_rate * dj_db

        print(f"Final cost :{self.cost_function(x, y, self.w, self.b):.4f}")
        return self.w, self.b

    def predict(self, x):
        """
        Predicts target values for new input data.

        Args:
            x (ndarray): Input data. shape (m,n) or (n,).

        Returns:
            ndarray : Predicted value(s) using the linear regression model (np.dot(w,x) + b).

        """
        return np.dot(x, self.w) + self.b


### Training the linear regression model

In [ ]:
x_train.shape

In [ ]:
x_train_np = x_train.values
y_train_np = y_train.values

x_test_np = x_test.values
y_test_np = y_test.values

In [ ]:
model = Multiple_LR_Model()
model.train_model(x_train_np,y_train_np)

In [ ]:
result = model.predict(x_test_np)

In [ ]:
print(result.shape)
print(y_test_np.shape)

In [ ]:
rmse = np.sqrt(np.mean((y_test_np - result)**2))
print(f"RMSE : {rmse}")

In [ ]:
mae = np.mean(np.abs(y_test_np - result))
print(f"MAE : {mae}")

In [ ]:
ssr  = np.sum((y_test_np - result)**2)
ssm = np.sum((y_test_np - y_test_np.mean())**2)
r2_score = 1 - (ssr/ssm)
print(r2_score)